<a href="https://colab.research.google.com/github/gunnsmart/skills/blob/main/AI_Image_Upscaler_Easy_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ AI Image Upscaler — ขยายภาพคมชัดด้วย AI (ฟรี)

โปรแกรมนี้ใช้ **Real-ESRGAN** ขยายภาพให้ใหญ่และคมชัดขึ้น 2-4 เท่า ใช้งานได้ฟรีบน Google Colab

### 📋 วิธีใช้ (ทำตาม 3 ขั้นตอน)

**ขั้นตอนที่ 1:** ตั้งค่า GPU ให้ฟรีและเร็วขึ้น
- ไปที่เมนู `Runtime` → `Change runtime type` → เลือก `T4 GPU` → กด `Save`

**ขั้นตอนที่ 2:** กดปุ่ม ▶️ ที่เซลล์แรกด้านล่าง เพื่อติดตั้งระบบ (รอประมาณ 1 นาที ทำครั้งเดียวต่อ session)

**ขั้นตอนที่ 3:** กดปุ่ม ▶️ ที่เซลล์ที่สอง เลือกขนาดที่ต้องการ แล้วอัปโหลดรูป (เลือกได้หลายรูปพร้อมกัน) — เสร็จแล้วไฟล์ ZIP จะดาวน์โหลดให้อัตโนมัติ

> 💡 ทำรูปชุดใหม่? แค่รันเซลล์ที่ 2 ซ้ำได้เลย ไม่ต้องรันเซลล์แรกใหม่

---


In [5]:

#@title 📥 ขั้นตอนที่ 1: เตรียมระบบ Real-ESRGAN (รันครั้งเดียว)
import os
import sys
import subprocess
import pathlib
import shutil

REPO = "/content/Real-ESRGAN"
BASICSR = "/content/BasicSR"


def run(cmd, check=True):
    print("▶", " ".join(cmd))

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout[-5000:])

        if check:
            raise RuntimeError(
                "Command failed: " + " ".join(cmd)
            )

    return result


# ============================================================
# 1. GPU
# ============================================================

gpu = run(["nvidia-smi"], check=False)

if gpu.returncode != 0:
    raise RuntimeError(
        "❌ ไม่พบ NVIDIA GPU\n\n"
        "กรุณาเลือก:\n"
        "Runtime → Change runtime type → T4 GPU"
    )

print("✅ GPU พร้อมใช้งาน")


# ============================================================
# 2. PyTorch
# ============================================================

import torch

print()
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPU     :", torch.cuda.get_device_name(0))

if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA ไม่พร้อมใช้งาน")


# ============================================================
# 3. Real-ESRGAN
# ============================================================

if not os.path.exists(REPO):

    print("\n📥 กำลังดาวน์โหลด Real-ESRGAN...")

    run([
        "git",
        "clone",
        "--depth", "1",
        "https://github.com/xinntao/Real-ESRGAN.git",
        REPO
    ])

else:
    print("\n📁 พบ Real-ESRGAN แล้ว")


# ============================================================
# 4. Runtime dependencies
# ============================================================

print("\n📦 ตรวจสอบ runtime dependencies...")

run([
    sys.executable,
    "-m", "pip", "install", "-q",
    "numpy",
    "opencv-python",
    "Pillow",
    "tqdm"
])

print("✅ runtime dependencies พร้อม")


# ============================================================
# 5. BasicSR
# ============================================================

if not os.path.exists(BASICSR):

    print("\n📥 กำลังดาวน์โหลด BasicSR...")

    run([
        "git",
        "clone",
        "--depth", "1",
        "https://github.com/XPixelGroup/BasicSR.git",
        BASICSR
    ])

else:
    print("\n📁 พบ BasicSR แล้ว")


# ============================================================
# 6. Generate BasicSR version.py
# ============================================================

VERSION_FILE = os.path.join(
    BASICSR,
    "VERSION"
)

if not os.path.exists(VERSION_FILE):
    raise RuntimeError(
        "❌ ไม่พบ BasicSR/VERSION"
    )

with open(VERSION_FILE) as f:
    basicsr_version = f.read().strip()

print()
print("BasicSR VERSION:", basicsr_version)

basicsr_version_py = os.path.join(
    BASICSR,
    "basicsr",
    "version.py"
)

os.makedirs(
    os.path.dirname(basicsr_version_py),
    exist_ok=True
)

with open(basicsr_version_py, "w") as f:
    f.write(
        f'__version__ = "{basicsr_version}"\n'
    )
    f.write(
        '__gitsha__ = "source"\n'
    )

print("✅ สร้าง basicsr/version.py แล้ว")


# ============================================================
# 7. Generate Real-ESRGAN version.py
# ============================================================

realesrgan_version_py = os.path.join(
    REPO,
    "realesrgan",
    "version.py"
)

os.makedirs(
    os.path.dirname(realesrgan_version_py),
    exist_ok=True
)

# Version ของ Real-ESRGAN repository
realesrgan_version = "0.3.0"

with open(realesrgan_version_py, "w") as f:
    f.write(
        f'__version__ = "{realesrgan_version}"\n'
    )
    f.write(
        '__gitsha__ = "source"\n'
    )

print(
    f"✅ สร้าง realesrgan/version.py "
    f"(v{realesrgan_version}) แล้ว"
)


# ============================================================
# 8. Torchvision compatibility
# ============================================================

print("\n🔧 ตรวจสอบ torchvision compatibility...")

patched = False

for file in pathlib.Path(BASICSR).rglob(
    "degradations.py"
):

    text = file.read_text()

    old = (
        "from torchvision.transforms.functional_tensor "
        "import rgb_to_grayscale"
    )

    new = (
        "from torchvision.transforms.functional "
        "import rgb_to_grayscale"
    )

    if old in text:

        file.write_text(
            text.replace(old, new)
        )

        print("✅ patched:", file)

        patched = True

if not patched:
    print("✅ ไม่พบ code ที่ต้อง patch")


# ============================================================
# 9. Python paths
# ============================================================

if BASICSR not in sys.path:
    sys.path.insert(0, BASICSR)

if REPO not in sys.path:
    sys.path.insert(0, REPO)

os.environ["PYTHONPATH"] = (
    BASICSR + ":" +
    REPO + ":" +
    os.environ.get("PYTHONPATH", "")
)


# ============================================================
# 10. Import tests
# ============================================================

print("\n🧪 ทดสอบ BasicSR...")

try:

    import basicsr

    print(
        f"✅ BasicSR {basicsr.__version__}"
    )

except Exception:

    import traceback
    traceback.print_exc()
    raise


print("\n🧪 ทดสอบ RRDBNet...")

try:

    from basicsr.archs.rrdbnet_arch import RRDBNet

    print("✅ RRDBNet OK")

except Exception:

    import traceback
    traceback.print_exc()
    raise


print("\n🧪 ทดสอบ Real-ESRGAN...")

try:

    from realesrgan import RealESRGANer

    print("✅ RealESRGANer OK")

except Exception:

    import traceback
    traceback.print_exc()
    raise


# ============================================================
# 11. Final
# ============================================================

print()
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("🎯 REAL-ESRGAN พร้อมใช้งาน")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print()
print("GPU        :", torch.cuda.get_device_name(0))
print("PyTorch    :", torch.__version__)
print("CUDA       :", torch.version.cuda)
print("BasicSR    :", basicsr.__version__)
print("RealESRGAN : 0.3.0")
print()
print("✅ ไม่ต้องรัน compatibility test เพิ่ม")
print("➡️ ไปขั้นตอนที่ 2 ได้เลย")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

▶ nvidia-smi
✅ GPU พร้อมใช้งาน

PyTorch : 2.11.0+cu128
CUDA    : 12.8
GPU     : Tesla T4

📁 พบ Real-ESRGAN แล้ว

📦 ตรวจสอบ runtime dependencies...
▶ /usr/bin/python3 -m pip install -q numpy opencv-python Pillow tqdm
✅ runtime dependencies พร้อม

📁 พบ BasicSR แล้ว

BasicSR VERSION: 1.4.2
✅ สร้าง basicsr/version.py แล้ว
✅ สร้าง realesrgan/version.py (v0.3.0) แล้ว

🔧 ตรวจสอบ torchvision compatibility...
✅ ไม่พบ code ที่ต้อง patch

🧪 ทดสอบ BasicSR...
✅ BasicSR 1.4.2

🧪 ทดสอบ RRDBNet...
✅ RRDBNet OK

🧪 ทดสอบ Real-ESRGAN...
✅ RealESRGANer OK

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 REAL-ESRGAN พร้อมใช้งาน
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

GPU        : Tesla T4
PyTorch    : 2.11.0+cu128
CUDA       : 12.8
BasicSR    : 1.4.2
RealESRGAN : 0.3.0

✅ ไม่ต้องรัน compatibility test เพิ่ม
➡️ ไปขั้นตอนที่ 2 ได้เลย
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:

#@title 🛠️ ขั้นตอนที่ 2: เลือกตั้งค่า แล้วกด Play เพื่ออัปโหลดรูปภาพ
import os
import sys
import time
import shutil
import zipfile
import traceback
import numpy as np
import torch

from PIL import Image, ImageOps
from google.colab import files


#@markdown ---
#@markdown ### 🎛️ 1. ต้องการให้รูปขยายใหญ่ขึ้นกี่เท่า:
ขยายขนาดรูป = "4 เท่า (แนะนำสำหรับภาพชัดเจน)" #@param ["2 เท่า (ขยายแบบพอดีๆ)", "4 เท่า (แนะนำสำหรับภาพชัดเจน)"]

#@markdown ### 🤖 2. รูปภาพเป็นประเภทไหน:
ประเภทของรูปภาพ = "ภาพถ่ายทิวทัศน์โฟโต้โทนชัดเจน (ภาพคน / ภาพถ่ายจริง)" #@param ["ภาพถ่ายทิวทัศน์โฟโต้โทนชัดเจน (ภาพคน / ภาพถ่ายจริง)", "ภาพวาดลายเส้นภาพการ์ตูนอนิเมะ 2D (ภาพกราฟิก / โลโก้ / ภาพวาด)"]

#@markdown ### 📂 3. นามสกุลไฟล์ที่ต้องการตอนจบ:
นามสกุลไฟล์ที่ต้องการ = "JPEG" #@param ["JPEG", "PNG"]

#@markdown ---

# ============================================================
# CONFIG
# ============================================================

SCALE = 4 if "4 เท่า" in ขยายขนาดรูป else 2

IS_ANIME = "อนิเมะ" in ประเภทของรูปภาพ

MODEL_NAME = (
    "RealESRGAN_x4plus_anime_6B"
    if IS_ANIME
    else
    "RealESRGAN_x4plus"
)

EXT = (
    ".jpg"
    if นามสกุลไฟล์ที่ต้องการ == "JPEG"
    else
    ".png"
)

REPO = "/content/Real-ESRGAN"
BASICSR = "/content/BasicSR"

INPUT_FOLDER = "/content/inputs"
OUTPUT_FOLDER = "/content/outputs"
WEIGHTS_FOLDER = os.path.join(REPO, "weights")


# ============================================================
# ตรวจ environment
# ============================================================

print("🔍 ตรวจสอบระบบ...")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ ไม่พบ GPU/CUDA\n"
        "กรุณาเปิด Runtime เป็น T4 GPU"
    )

print("✅ GPU     :", torch.cuda.get_device_name(0))
print("✅ PyTorch :", torch.__version__)
print("✅ CUDA    :", torch.version.cuda)


# ============================================================
# Import
# ============================================================

sys.path.insert(0, BASICSR)
sys.path.insert(0, REPO)

print("\n🔧 โหลด Real-ESRGAN...")

try:

    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    print("✅ Real-ESRGAN API พร้อม")

except Exception:

    print("❌ ไม่สามารถโหลด Real-ESRGAN")

    traceback.print_exc()

    raise


# ============================================================
# Model architecture
# ============================================================

print("\n🤖 Model:", MODEL_NAME)

if MODEL_NAME == "RealESRGAN_x4plus":

    model = RRDBNet(
        num_in_ch=3,
        num_out_ch=3,
        num_feat=64,
        num_block=23,
        num_grow_ch=32,
        scale=4
    )

    MODEL_URL = (
        "https://github.com/xinntao/Real-ESRGAN/releases/"
        "download/v0.2.5.0/RealESRGAN_x4plus.pth"
    )

else:

    model = RRDBNet(
        num_in_ch=3,
        num_out_ch=3,
        num_feat=64,
        num_block=6,
        num_grow_ch=32,
        scale=4
    )

    MODEL_URL = (
        "https://github.com/xinntao/Real-ESRGAN/releases/"
        "download/v0.2.5.0/RealESRGAN_x4plus_anime_6B.pth"
    )


# ============================================================
# Download model
# ============================================================

os.makedirs(
    WEIGHTS_FOLDER,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    WEIGHTS_FOLDER,
    MODEL_NAME + ".pth"
)

if not os.path.exists(MODEL_PATH):

    print("\n📥 กำลังดาวน์โหลด model...")

    import subprocess

    result = subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "-O",
            MODEL_PATH,
            MODEL_URL
        ]
    )

    if result.returncode != 0:

        if os.path.exists(MODEL_PATH):
            os.remove(MODEL_PATH)

        raise RuntimeError(
            "❌ ดาวน์โหลด model ไม่สำเร็จ"
        )

else:

    print("✅ พบ model แล้ว")


# ============================================================
# Upload ก่อน เพื่อเลือก tile ตามภาพ
# ============================================================

for folder in [
    INPUT_FOLDER,
    OUTPUT_FOLDER
]:

    if os.path.exists(folder):
        shutil.rmtree(folder)

    os.makedirs(
        folder,
        exist_ok=True
    )


print("\n📤 เลือกรูปภาพ")
print("สามารถเลือกหลายไฟล์พร้อมกันได้")

uploaded = files.upload()

if not uploaded:

    print("❌ ไม่ได้เลือกรูปภาพ")

    raise RuntimeError(
        "ไม่มีไฟล์ input"
    )


# ============================================================
# Move uploads
# ============================================================

for filename in uploaded:

    src = os.path.join(
        "/content",
        filename
    )

    dst = os.path.join(
        INPUT_FOLDER,
        filename
    )

    if os.path.exists(src):

        shutil.move(
            src,
            dst
        )


images = sorted([
    f
    for f in os.listdir(INPUT_FOLDER)
    if f.lower().endswith(
        (
            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".bmp",
            ".tif",
            ".tiff"
        )
    )
])

if not images:

    raise RuntimeError(
        "❌ ไม่พบไฟล์รูปภาพที่รองรับ"
    )

print(
    f"\n📥 รับรูปสำเร็จ {len(images)} ภาพ"
)


# ============================================================
# Determine maximum image size
# ============================================================

image_info = []

for filename in images:

    path = os.path.join(
        INPUT_FOLDER,
        filename
    )

    try:

        with Image.open(path) as im:

            w, h = im.size

            image_info.append(
                (filename, w, h)
            )

    except Exception:

        print(
            f"⚠️ อ่านไฟล์ไม่ได้: {filename}"
        )


if not image_info:

    raise RuntimeError(
        "ไม่มีรูปที่สามารถเปิดได้"
    )


max_pixels = max(
    w * h
    for _, w, h in image_info
)


# ============================================================
# Dynamic tile
# ============================================================

if max_pixels > 20_000_000:

    TILE = 192

elif max_pixels > 12_000_000:

    TILE = 256

elif max_pixels > 6_000_000:

    TILE = 384

else:

    TILE = 512


print()
print("⚙️ GPU configuration")
print("   Tile :", TILE)
print("   FP16 : ON")
print("   Scale:", SCALE, "x")


# ============================================================
# Create Real-ESRGAN
# ============================================================

print("\n⚙️ กำลังโหลด model ลง GPU...")

try:

    upsampler = RealESRGANer(
        scale=4,
        model_path=MODEL_PATH,
        model=model,
        tile=TILE,
        tile_pad=10,
        pre_pad=0,
        half=True,
        gpu_id=0
    )

except Exception:

    print("❌ ไม่สามารถโหลด model ลง GPU")

    traceback.print_exc()

    raise


print("✅ Model พร้อมใช้งาน")


# ============================================================
# Processing
# ============================================================

success_count = 0
failed = []

start_time = time.time()


for idx, filename in enumerate(
    images,
    1
):

    input_path = os.path.join(
        INPUT_FOLDER,
        filename
    )

    stem = os.path.splitext(
        filename
    )[0]

    output_path = os.path.join(
        OUTPUT_FOLDER,
        stem + EXT
    )

    print()
    print(
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"
    )

    print(
        f"🖼️ [{idx}/{len(images)}] {filename}"
    )

    try:

        # ----------------------------------------------------
        # Load
        # ----------------------------------------------------

        with Image.open(input_path) as im:

            # เคารพ EXIF orientation
            im = ImageOps.exif_transpose(im)

            im = im.convert("RGB")

            w, h = im.size

            print(
                f"   Input  : {w:,} × {h:,}"
            )

            image_np = np.asarray(
                im,
                dtype=np.uint8
            )


        # ----------------------------------------------------
        # Upscale
        # ----------------------------------------------------

        print(
            f"   🤖 AI Upscale {SCALE}x..."
        )

        output_np, _ = upsampler.enhance(
            image_np,
            outscale=SCALE
        )


        # ----------------------------------------------------
        # Output
        # ----------------------------------------------------

        result = Image.fromarray(
            output_np
        )

        ow, oh = result.size

        print(
            f"   Output : {ow:,} × {oh:,}"
        )


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        if EXT == ".jpg":

            result.save(
                output_path,
                "JPEG",
                quality=95,
                subsampling=0,
                optimize=True,
                dpi=(300, 300)
            )

        else:

            result.save(
                output_path,
                "PNG",
                optimize=True,
                dpi=(300, 300)
            )


        print("   ✅ สำเร็จ")

        success_count += 1

        del image_np
        del output_np
        del result

        torch.cuda.empty_cache()


    except Exception:

        print("   ❌ ERROR")

        traceback.print_exc()

        failed.append(
            filename
        )

        torch.cuda.empty_cache()


# ============================================================
# Summary
# ============================================================

elapsed = time.time() - start_time

print()
print(
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"
)

print("📊 สรุปผลการประมวลผล")

print(
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"
)

print(
    f"จำนวนทั้งหมด : {len(images)}"
)

print(
    f"สำเร็จ       : {success_count}"
)

print(
    f"ล้มเหลว      : {len(failed)}"
)

print(
    f"เวลา         : {elapsed:.1f} วินาที"
)


if failed:

    print("\n⚠️ ไฟล์ที่ทำไม่สำเร็จ:")

    for filename in failed:

        print(
            " -",
            filename
        )


# ============================================================
# ZIP
# ============================================================

if success_count > 0:

    zip_name = (
        f"upscaled_{SCALE}x.zip"
    )

    zip_path = os.path.join(
        "/content",
        zip_name
    )

    if os.path.exists(zip_path):

        os.remove(zip_path)


    print(
        "\n📦 กำลังสร้าง ZIP..."
    )

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        for filename in os.listdir(
            OUTPUT_FOLDER
        ):

            path = os.path.join(
                OUTPUT_FOLDER,
                filename
            )

            if os.path.isfile(path):

                z.write(
                    path,
                    filename
                )


    print(
        f"✅ ZIP พร้อม: {zip_name}"
    )

    print(
        "📥 กำลังดาวน์โหลด..."
    )

    files.download(
        zip_path
    )

else:

    print(
        "\n❌ ไม่มีไฟล์ที่ประมวลผลสำเร็จ"
    )

🔍 ตรวจสอบระบบ...
✅ GPU     : Tesla T4
✅ PyTorch : 2.11.0+cu128
✅ CUDA    : 12.8

🔧 โหลด Real-ESRGAN...
✅ Real-ESRGAN API พร้อม

🤖 Model: RealESRGAN_x4plus
✅ พบ model แล้ว

📤 เลือกรูปภาพ
สามารถเลือกหลายไฟล์พร้อมกันได้
